# Brain Wave Band Power

**Dataset**: PhysioNet Auditory EEG (Abo Alzahab et al., 2021)  
**Channels**: P4, Cz, F8, T7  
**Sampling rate**: 200 Hz  
**Subject**: 1

---

## Overview

We use Welch's method to estimate the power spectral density of the signal, then integrate the power in each brain wave band (delta, theta, alpha, beta, gamma) and compare them as percentages.

## Expected outputs

- A PSD curve showing power distribution across frequencies
- A bar chart comparing relative power in each band
- Concentration in low bands (delta, theta) due to the 1/f law

## Key parameters

| Parameter | Value | Meaning |
| --- | --- | --- |
| Channel | P4 | Parietal region |
| Sampling rate | 200 Hz | One sample every 5 ms |
| nperseg | 1024 | Welch window size |
| Bands | 5 | delta, theta, alpha, beta, gamma |


## 1. Install dependencies


In [ ]:
!pip install scipy numpy plotly wfdb pywt


## 2. Clone repo and download data

We download only subject 1 (`--subjects 1`) to speed up the experiment in Colab.


In [ ]:
import os
if not os.path.exists('python-EEG-Arabic-Resources'):
    !git clone https://github.com/NibrasAz7/python-EEG-Arabic-Resources.git
os.chdir('python-EEG-Arabic-Resources')


In [ ]:
from pathlib import Path
data_dir = Path('data/local')
if not data_dir.exists() or not any(data_dir.glob('*.dat')):
    !python data/download_local.py --output data/local --subjects 1


## 3. Load the EEG signal

We load subject 1, experiment 1, session 2, channel P4 (parietal region).


In [ ]:
import numpy as np
from utils.eeg_loader import load_local_eeg

timestamps, eeg_data, ch_names = load_local_eeg(
    data_dir='data/local', subject=1, experiment=1, session=2
)
channel_data = eeg_data[:, 0]  # P4 channel
fs = 200  # Sampling rate (Hz)

print(f'Channels: {ch_names}')
print(f'Signal length: {len(channel_data)} samples ({len(channel_data)/fs:.1f} seconds)')


## 4. Compute band power

We use `scipy.signal.welch` to estimate the power spectral density. Then we integrate the power in each frequency band using `np.trapezoid`.

We apply the clean_signal function to remove noise via 1-45 Hz bandpass and 50 Hz notch filter.

In [ ]:
from scipy.signal import welch, butter, filtfilt, iirnotch

def clean_signal(signal, fs=200, low=1.0, high=45.0, notch_freq=50.0):
    b_bp, a_bp = butter(4, [low / (fs / 2), high / (fs / 2)], btype='band')
    filtered = filtfilt(b_bp, a_bp, signal)
    b_notch, a_notch = iirnotch(notch_freq, 30.0, fs=fs)
    filtered = filtfilt(b_notch, a_notch, filtered)
    return filtered

channel_data = clean_signal(channel_data, fs=fs)

freqs, psd = welch(channel_data, fs=fs, nperseg=1024)

BANDS = [
    ('Delta', 0.5, 4),
    ('Theta', 4, 8),
    ('Alpha', 8, 13),
    ('Beta', 13, 30),
    ('Gamma', 30, 80),
]

band_powers = {}
for name, fmin, fmax in BANDS:
    band_mask = (freqs >= fmin) & (freqs <= fmax)
    power = np.trapezoid(psd[band_mask], freqs[band_mask])
    band_powers[name] = power

total_power = sum(band_powers.values())
relative_powers = {k: v / total_power * 100 for k, v in band_powers.items()}
for name, power in relative_powers.items():
    print(f'{name}: {power:.1f}%')

## 5. Interactive plot

**What to look for:**

- The PSD curve shows power distribution on a logarithmic scale
- Colored shading marks the five brain wave bands
- The bar chart compares relative power as percentages



In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

BAND_COLORS = ['green', 'blue', 'orange', 'red', 'purple']

fig = make_subplots(rows=2, cols=1, shared_xaxes=False,
                    subplot_titles=('Power Spectral Density - Channel P4',
                                    'Relative Band Power - Channel P4'))
fig.add_trace(go.Scatter(x=freqs, y=psd, name='PSD',
                         line=dict(color='black', width=1)), row=1, col=1)
for (name, fmin, fmax), color in zip(BANDS, BAND_COLORS):
    fig.add_vrect(x0=fmin, x1=fmax, fillcolor=color, opacity=0.1,
                  line_width=0, row=1, col=1)
fig.update_xaxes(range=[0, 80], row=1, col=1)
fig.update_yaxes(type='log', row=1, col=1)

names = list(relative_powers.keys())
values = list(relative_powers.values())
fig.add_trace(go.Bar(x=names, y=values, marker_color=BAND_COLORS,
                     name='Relative Power'), row=2, col=1)

fig.update_layout(height=800, title_text='Brain Wave Band Power - Channel P4',
                  xaxis_title='Frequency (Hz)', xaxis2_title='Band',
                  yaxis_title='PSD (uV^2/Hz)', yaxis2_title='Relative Power (%)',
                  showlegend=False)
fig.show()


## What did we learn?

- Welch's method efficiently estimates the power spectral density of the signal
- Relative power makes it easier to compare bands and removes the effect of absolute amplitude
- In raw EEG data, power concentrates in low bands (1/f law)
- It is recommended to apply a bandpass filter before computing power to remove artifacts

